# Hyperparameter Tuning with Optuna
Picks up from `07_modelling_v2.ipynb` — requires the same data pipeline to have run first.

**Current baseline:** V7 RMSPE = 0.1140 (`lr=0.01, leaves=127, min_child_samples=30`)

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# ── Data (same as modelling notebook) ──────────────────────────────────────
df = pd.read_csv('../data/processed/train_features.csv')

TARGET   = 'Sales'
FEATURES = [c for c in df.columns if c not in ('Sales', 'Customers')]

df = df.sort_values(['Year', 'Week']).reset_index(drop=True)

split_idx = int(len(df) * 0.85)
train_df  = df.iloc[:split_idx].copy()
val_df    = df.iloc[split_idx:].copy()

# ── Engineered features (V3 set from modelling notebook) ───────────────────
store_mean = train_df.groupby('Store')['Sales'].mean()
train_df['StoreMeanSales'] = train_df['Store'].map(store_mean)
val_df['StoreMeanSales']   = val_df['Store'].map(store_mean)

dow_mean = train_df.groupby(['Store', 'DayOfWeek'])['Sales'].mean()
train_df['StoreDowMean'] = train_df.set_index(['Store', 'DayOfWeek']).index.map(dow_mean)
val_df['StoreDowMean']   = val_df.set_index(['Store', 'DayOfWeek']).index.map(dow_mean)

easter_dates = pd.to_datetime(["2013-03-31", "2014-04-20", "2015-04-05"])

def days_to_nearest_easter(row):
    date  = pd.Timestamp(year=int(row["Year"]), month=int(row["Month"]), day=int(row["Day"]))
    diffs = [(date - e).days for e in easter_dates]
    return min(diffs, key=abs)

train_df["DaysToEaster"] = train_df.apply(days_to_nearest_easter, axis=1)
val_df["DaysToEaster"]   = val_df.apply(days_to_nearest_easter, axis=1)

FEATURES_TUNE = FEATURES + ['StoreMeanSales', 'StoreDowMean', 'DaysToEaster']

# ── Log-transform target ───────────────────────────────────────────────────
train_df['LogSales'] = np.log1p(train_df['Sales'])
val_df['LogSales']   = np.log1p(val_df['Sales'])
LOG_TARGET = 'LogSales'

X_train = train_df[FEATURES_TUNE]
y_train = train_df[LOG_TARGET]
X_val   = val_df[FEATURES_TUNE]
y_val   = val_df[LOG_TARGET]

lgb_train_base = lgb.Dataset(X_train, label=y_train)
lgb_val_base   = lgb.Dataset(X_val,   label=y_val, reference=lgb_train_base)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Features: {len(FEATURES_TUNE)}')

Train: 717,687 | Val: 126,651 | Features: 21


In [4]:
def rmspe(y_true, y_pred):
    mask = y_true != 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

In [5]:
# ── Optuna objective ───────────────────────────────────────────────────────
# We keep learning_rate fixed at 0.02 and rely on early stopping.
# This lets Optuna focus on the structural / regularisation params,
# which have a far bigger impact and are faster to explore.

def objective(trial): 
    params = {
        'objective':        'regression',
        'metric':           'rmse',
        'verbose':          -1,
        'learning_rate':    0.02, 
        'bagging_freq':      1, 

        # Tree structure
        'num_leaves': trial.suggest_int('num_leaves', 63, 255),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),

        # Stochastic regularisation
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),

        # L1 / L2
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),

        # Min gain to split — helps prune noisy splits
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0)
    }

    # Fresh datasets each trial
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    callbacks = [
        lgb.early_stopping(stopping_rounds=100, verbose=False),
        lgb.log_evaluation(period=99999)          # silent
    ]

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=5000,
        valid_sets=[dval],
        callbacks=callbacks
    )

    log_preds = model.predict(X_val)
    preds = np.maximum(np.expm1(log_preds), 0)
    score = rmspe(val_df['Sales'].values, preds)

    # Store best iteration for later inspectation
    trial.set_user_attr('best_iteration', model.best_iteration)
    return score



In [6]:
# ── Run the study ──────────────────────────────────────────────────────────
# n_trials=50 Or
# Increase to 100  — results tend to plateau after ~60 trials.

study = optuna.create_study(direction='minimize', study_name='lgbm_rossmann', sampler=optuna.samplers.TPESampler(seed=42))

# Seed with known good config so Optuna starts from a strong baseline
study.enqueue_trial({
    'num_leaves':        127,
    'max_depth':         10,
    'min_child_samples': 30,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      1,
    'lambda_l1':         0.01,
    'lambda_l2':         0.01,
    'min_split_gain':    0.0,
})

N_TRIALS = 50 

study.optimize(
    objective, 
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print(f'\nBest RMSPE: {study.best_value:.4f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f' {k}: {v}')
print(f'  best_iteration: {study.best_trial.user_attrs["best_iteration"]}')

  0%|          | 0/50 [00:00<?, ?it/s]


Best RMSPE: 0.1150
Best params:
 num_leaves: 227
 max_depth: 10
 min_child_samples: 84
 feature_fraction: 0.6995779135100955
 bagging_fraction: 0.80551501219963
 lambda_l1: 0.0070337240611416645
 lambda_l2: 0.053401411348692966
 min_split_gain: 0.0002388763976012647
  best_iteration: 4370


In [7]:
# ── Results summary ──────────────────────────────────────────────────
results_df = study.trials_dataframe()
results_df = results_df.sort_values('value').reset_index(drop=True)
print('Top 10 trials:')
print(results_df[['number', 'value', 'params_num_leaves', 'params_min_child_samples',
                   'params_feature_fraction', 'params_lambda_l1', 'params_lambda_l2']].head(10).to_string())

Top 10 trials:
   number     value  params_num_leaves  params_min_child_samples  params_feature_fraction  params_lambda_l1  params_lambda_l2
0      22  0.114986                227                        84                 0.699578          0.007034          0.053401
1       0  0.115325                127                        30                 0.800000          0.010000          0.010000
2      13  0.115656                211                        43                 0.735236          0.015298          0.016015
3      14  0.116047                235                        60                 0.699947          0.121893          0.016074
4      42  0.116809                113                        95                 0.675433          0.010141          0.013505
5      26  0.116963                160                        89                 0.687996          0.010317          0.043177
6      36  0.117523                 98                        27                 0.639961          0.01

In [ ]:
# ── Retrain final model with best params + slower LR for max accuracy ──────

best = study.best_params.copy()

final_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbose': -1,
    'learning_rate': 0.01,          # slower LR for final polish
    **best
}

dtrain_final = lgb.Dataset(X_train, label=y_train)
dval_final = lgb.Dataset(X_val, label=y_val, reference=dtrain_final)

callbacks_final = [
    lgb.early_stopping(stopping_rounds=200, verbose=False),
    lgb.log_evaluation(period=1000)
]

model_tuned = lgb.train(
    final_params,
    dtrain_final,
    num_boost_round=20000,
    valid_sets=[dval_final],
    callbacks=callbacks_final
)

log_preds_tuned = model_tuned.predict(X_val)
preds_tuned = np.maximum(np.expm1(log_preds_tuned), 0)
rmspe_tuned = rmspe(val_df['Sales'].values, preds_tuned)

print(f'\n------- Final Results ------')
print(f'V7 baseline (manual):      0.1140')
print(f'Optuna best (lr=0.02):     {study.best_value:.4f}')
print(f'Final model (lr=0.01):     {rmspe_tuned:.4f}')
print(f'Best iteration:            {model_tuned.best_iteration}')

[1000]	valid_0's rmse: 0.128658
[2000]	valid_0's rmse: 0.124032
[3000]	valid_0's rmse: 0.121376
[4000]	valid_0's rmse: 0.119759
[5000]	valid_0's rmse: 0.118629
[6000]	valid_0's rmse: 0.11777
[7000]	valid_0's rmse: 0.117167
[8000]	valid_0's rmse: 0.116684
[9000]	valid_0's rmse: 0.116419
[10000]	valid_0's rmse: 0.116166

------- Final Results ------
V7 baseline (manual):      0.1297
Optuna best (lr=0.02):     0.1150
Final model (lr=0.01):     0.1148
Best iteration:            9814


In [9]:
# ── Save the tuned model ───────────────────────────────────────────────────
model_tuned.save_model('../models/lgbm_tuned.txt')
print('Model saved to ../models/lgbm_tuned.txt')

# Also save the params for reproducibility
import json
with open('../models/lgbm_tuned_params.json', 'w') as f:
    json.dump(final_params, f, indent=2)
print('Params saved to ../models/lgbm_tuned_params.json')

Model saved to ../models/lgbm_tuned.txt
Params saved to ../models/lgbm_tuned_params.json
